# Kopling Aroma dan Agen pada Lebah Madu

**ID proyek:** `O005-LEGA-V101-PRJ10`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Apakah aturan gerak yang menggabungkan gradien aroma dan derau cukup untuk menghasilkan akumulasi agen di dekat sumber?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082210
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Aroma adalah medan Gaussian statis; tiap lebah bergerak mengikuti gradien lokal ditambah gerak acak kecil; domain dibatasi dan sumber tetap.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
n_bees, n_steps = 64, 100
source = np.array([0.78, 0.72])
positions = rng.uniform(0.05, 0.95, (n_bees, 2))
paths = np.empty((n_steps + 1, n_bees, 2))
paths[0] = positions
initial_distance = np.linalg.norm(positions - source, axis=1)

def scent(points):
    return np.exp(-np.sum((points - source) ** 2, axis=-1) / 0.10)

for step in range(n_steps):
    displacement = source - positions
    distance = np.linalg.norm(displacement, axis=1, keepdims=True)
    direction = displacement / np.maximum(distance, 1e-12)
    coupling = 0.35 + 0.65 * scent(positions)[:, None]
    noise = rng.normal(0.0, 0.0035, positions.shape)
    positions = np.clip(positions + 0.013 * coupling * direction + noise, 0.0, 1.0)
    paths[step + 1] = positions
final_distance = np.linalg.norm(positions - source, axis=1)
xg = np.linspace(0.0, 1.0, 80)
Xg, Yg = np.meshgrid(xg, xg)
scent_field = scent(np.stack([Xg, Yg], axis=-1))


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert np.all((positions >= 0.0) & (positions <= 1.0))
assert float(np.median(final_distance)) < 0.45 * float(np.median(initial_distance))
assert np.min(scent_field) >= 0.0 and np.max(scent_field) > 0.99


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
contour = axes[0].contourf(Xg, Yg, scent_field, levels=15, cmap="YlOrBr")
for bee in range(0, n_bees, 4):
    axes[0].plot(paths[:, bee, 0], paths[:, bee, 1], color="tab:blue", alpha=0.5, linewidth=0.8)
axes[0].scatter(*source, marker="*", s=120, color="red", label="sumber")
axes[0].set(title="Lintasan di medan aroma", xlim=(0, 1), ylim=(0, 1), aspect="equal")
axes[0].legend(fontsize=8)
fig.colorbar(contour, ax=axes[0], label="intensitas")
axes[1].hist(initial_distance, bins=12, alpha=0.65, label="awal")
axes[1].hist(final_distance, bins=12, alpha=0.65, label="akhir")
axes[1].set(xlabel="jarak ke sumber", ylabel="jumlah lebah", title="Perubahan jarak")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Tidak ada dinamika turbulensi aroma, komunikasi tarian, penghindaran tumbukan, memori, atau heterogenitas sensorik.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
